# Read data

In [4]:
import pandas as pd
df = pd.read_csv('../../data/processed/products.csv')
i_df = pd.read_csv('../../data/processed/mock_interactions.csv')


In [ ]:
df.describe()

,id,original_price,price,review_count,rating_average,favourite_count,date_created,number_of_images,vnd_cashback,quantity_sold,popularity_weight
count,4.157600e+04,4.157600e+04,4.157600e+04,41576.000000,41576.000000,41576.0,41576.000000,41576.000000,41576.000000,41576.000000,41576.000000
mean,1.465187e+08,3.049790e+05,2.723589e+05,3.112084,1.381545,0.0,711.186117,5.969093,1431.424836,17.776939,1.857765
std,6.365429e+07,8.518988e+05,7.630026e+05,18.231025,2.105277,0.0,6275.529624,3.455423,6388.980325,168.379391,1.218802
min,1.599010e+05,0.000000e+00,0.000000e+00,0.000000,0.000000,0.0,0.000000,1.000000,0.000000,0.000000,1.000000
25%,1.020968e+08,4.000000e+04,3.900000e+04,0.000000,0.000000,0.0,396.750000,4.000000,0.000000,0.000000,1.000000
50%,1.523477e+08,9.000000e+04,8.500000e+04,0.000000,0.000000,0.0,616.000000,6.000000,0.000000,0.000000,1.000000
75%,1.935654e+08,2.400000e+05,2.100000e+05,1.000000,4.000000,0.0,831.000000,8.000000,0.000000,5.000000,2.406329
max,2.633026e+08,1.980000e+07,1.980000e+07,969.000000,5.000000,0.0,738076.000000,70.000000,167720.000000,24847.000000,9.399036


In [ ]:
i_df.describe()

    user_id    item_id event_type            timestamp
0  USR_1341   66558475       view  2026-06-24 18:41:00
1  USR_2732   74541439       view  2026-06-03 17:02:00
2  USR_1255  188996612       view  2026-06-02 13:19:00
3  USR_2088  107801586       view  2026-06-30 09:07:00
4  USR_1476   94038063       view  2026-06-30 04:18:00


In [5]:
import numpy as np
event_counts = i_df.groupby(['user_id', 'item_id', 'event_type']).size().unstack(fill_value=0).reset_index()

for col in ['view', 'cart', 'purchase']:
    if col not in event_counts.columns:
        event_counts[col] = 0

a = 1  # Trọng số cho View
b = 3  # Trọng số cho Cart
g = 5  # Trọng số cho Purchase


event_counts['implicit_score'] = (
    np.log1p(event_counts['view']) * a + 
    np.log1p(event_counts['cart']) * b + 
    np.log1p(event_counts['purchase']) * g
)

print(event_counts[['user_id', 'item_id', 'view', 'cart', 'purchase', 'implicit_score']].head())

event_type   user_id   item_id  view  cart  purchase  implicit_score
0           USR_0001  68117476     1     0         0        0.693147
1           USR_0001  68767421     4     0         0        1.609438
2           USR_0001  73297007     3     0         0        1.386294
3           USR_0001  76193509     2     0         0        1.098612
4           USR_0001  79228515     3     0         0        1.386294


In [6]:
import scipy.sparse as sparse
import pickle

# 1. Chuyển đổi ID sang Index liên tục (Categorical encoding)
user_cat = event_counts['user_id'].astype("category")
item_cat = event_counts['item_id'].astype("category")

event_counts['user_index'] = user_cat.cat.codes
event_counts['item_index'] = item_cat.cat.codes

# 2. Tạo từ điển Mapping để sau này dịch ngược từ Index ra ID gốc
# (Rất quan trọng để hiển thị đúng sản phẩm lên giao diện web)
user_mapping = dict(enumerate(user_cat.cat.categories))
item_mapping = dict(enumerate(item_cat.cat.categories))
item_id_to_index = {v: k for k, v in item_mapping.items()} # Lookup ngược

# 3. Khởi tạo Sparse Matrix (Dạng CSR - Compressed Sparse Row)
sparse_user_item = sparse.csr_matrix(
    (event_counts['implicit_score'], (event_counts['user_index'], event_counts['item_index']))
)

# 4. Kiểm tra độ thưa thớt (Sparsity)
matrix_size = sparse_user_item.shape[0] * sparse_user_item.shape[1]
num_interactions = len(sparse_user_item.nonzero()[0])
sparsity = 100 * (1 - (num_interactions / matrix_size))

print(f"Kích thước ma trận (Users, Items): {sparse_user_item.shape}")
print(f"Độ thưa thớt của ma trận: {sparsity:.4f}%")

Kích thước ma trận (Users, Items): (3000, 16177)
Độ thưa thớt của ma trận: 99.8945%


In [ ]:
import implicit
import os

# 1. Khởi tạo mô hình ALS
# Bạn có thể tinh chỉnh các Hyperparameter này (factors, regularization, iterations)
model = implicit.als.AlternatingLeastSquares(
    factors=50,             # Số lượng đặc trưng ẩn (Latent features)
    regularization=0.1,     # Tránh Overfitting
    iterations=20,          # Số vòng lặp huấn luyện
    random_state=42
)

# 2. Huấn luyện mô hình với ma trận vừa tạo
print("Đang huấn luyện mô hình ALS...")
model.fit(sparse_user_item)
print("Huấn luyện thành công!")

# 3. Lưu mô hình và file Mapping lại để dùng cho API gợi ý
MODEL_DIR = '../../models'
os.makedirs(MODEL_DIR, exist_ok=True)

with open(os.path.join(MODEL_DIR, 'als_model.pkl'), 'wb') as f:
    pickle.dump(model, f)

with open(os.path.join(MODEL_DIR, 'user_id_map.pkl'), 'wb') as f:
    pickle.dump(user_mapping, f)
    
with open(os.path.join(MODEL_DIR, 'item_id_map.pkl'), 'wb') as f:
    pickle.dump(item_mapping, f)

print(f"Đã lưu các file model mới tinh vào thư mục {MODEL_DIR}")

d:\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\miniconda3\Lib\site-packages\implicit\cpu\als.py:96: RuntimeWarning: Intel MKL BLAS is configured to use 8 threads. It is highly recommended to disable its internal threadpool by setting the environment variable 'MKL_NUM_THREADS=1' or by callng 'threadpoolctl.threadpool_limits(1, "blas")'. Having MKL use a threadpool can lead to severe performance issues
  check_blas_config()


Đang huấn luyện mô hình ALS...


100%|██████████| 20/20 [00:00<00:00, 115.44it/s]

Huấn luyện thành công!


In [9]:
import pandas as pd
import numpy as np

products_df = pd.read_csv('../../data/processed/products.csv')

# Ép kiểu sẵn ở vòng ngoài để tối ưu tốc độ
products_df['id'] = products_df['id'].astype(int)
user_id_to_index = {v: k for k, v in user_mapping.items()}

# ==========================================
# 2. HÀM TẠO GỢI Ý (INFERENCE & RERANKING)
# ==========================================
def get_recommendations(user_id, weight_rating=0.2, top_n=10):
    # Kiểm tra User có tồn tại không
    if user_id not in user_id_to_index:
        return "User chưa từng có tương tác (Cold-start). Hệ thống sẽ dùng thuật toán Popularity để gợi ý."
    
    user_idx = user_id_to_index[user_id]
    
    # 1. Candidate Generation: Lấy Top 50 từ ALS
    item_indices, als_scores = model.recommend(user_idx, sparse_user_item[user_idx], N=50)
    recommended_item_ids = [item_mapping[i] for i in item_indices]
    
    recs_df = pd.DataFrame({
        'item_id': recommended_item_ids,
        'raw_als_score': als_scores
    })
    
    # Ép kiểu an toàn trước khi join
    recs_df['item_id'] = recs_df['item_id'].astype(int)
    
    # 2. Reranking Data Preparation: Join với bảng sản phẩm
    recs_df = recs_df.merge(
        products_df[['id', 'name', 'sub_category', 'rating_average', 'review_count']], 
        left_on='item_id', right_on='id', how='left'
    )
    
    # 3. Chuẩn hóa điểm số (Min-Max cho ALS, Clip cho Rating)
    min_als = recs_df['raw_als_score'].min()
    max_als = recs_df['raw_als_score'].max()
    
    if max_als > min_als:
        recs_df['als_norm'] = (recs_df['raw_als_score'] - min_als) / (max_als - min_als)
    else:
        recs_df['als_norm'] = 0.5 
        
    # Xử lý ngoại lệ rating vượt mức 5.0 bằng np.clip
    recs_df['rating_norm'] = np.clip(recs_df['rating_average'].fillna(0) / 5.0, 0, 1)
    
    # 4. Tính Final Score
    recs_df['final_score'] = ((1 - weight_rating) * recs_df['als_norm']) + (weight_rating * recs_df['rating_norm'])
    
    # 5. Sắp xếp và xuất kết quả Top N
    final_recs = recs_df.sort_values(by='final_score', ascending=False).head(top_n)
    
    display_cols = ['item_id', 'name', 'sub_category', 'rating_average', 'review_count', 'als_norm', 'rating_norm', 'final_score']
    return final_recs[display_cols].reset_index(drop=True)

# Test thử kết quả với trọng số Rating chiếm 20%
ket_qua = get_recommendations(user_id='USR_0001', weight_rating=0.2, top_n=10)
print(ket_qua)

     item_id                                               name sub_category  \
0  257326416  Keo Dán Giày Nhiệt XIMO Trong Suốt Siêu Dính D...    men_shoes   
1  134739207  Giày da nam, giày oxford công sở G103 - Da bò ...    men_shoes   
2   72570436              Dép Nam Quai Ngang Kẻ SUPERSTAR DP65    men_shoes   
3  250111755       Túi Đeo Chéo Nam 4U Da Tổng Hợp Cao Cấp D267     men_bags   
4   71890733  Dép quai ngang nam nữ đế đúc DUWA T464KG - Hàn...    men_shoes   
5   52281671  Lót giày nam PETTINO tăng chiều cao 1.5cm - 3....    men_shoes   
6   84865829  Dép nam da thật đế cao su đúc siêu biền - chốn...    men_shoes   
7   73505299  Giày sneaker thể thao nam thời trang buộc dây ...    men_shoes   
8  116692721  Dép cao su nam phong cách thể thao sport 2021 D01    men_shoes   
9   79691242  Ví Thẻ Tín Dụng, Bóp Đựng Thẻ Tín Dụng, Bóp Hộ...     men_bags   

   rating_average  review_count  als_norm  rating_norm  final_score  
0             4.6           926  1.000000        